In [ ]:
#!/usr/bin/env python3
"""
╔══════════════════════════════════════════════════════════════════════════════╗
║  KAGGLE TRAINING SCRIPT — YOLOv8 RGBD (4-Channel) Detection               ║
║                                                                            ║
║  Pipeline:                                                                 ║
║    1. Install ultralytics_RGBD fork (panchasan/ultralytics_RGBD)           ║
║    2. Download dataset from Roboflow (4k images, 75/25 split)              ║
║    3. Generate depth maps using Depth Anything V2 (ViT-S)                  ║
║    4. Merge RGB + Depth → 4-channel TIFF images                            ║
║    5. Train YOLOv8x-RGBD on 2× T4 GPUs                                    ║
║                                                                            ║
║  Usage: Run all cells in a Kaggle notebook with GPU T4 ×2 accelerator.     ║
╚══════════════════════════════════════════════════════════════════════════════╝
"""
# =============================================================================
# CELL 1 — Install Dependencies
# =============================================================================
# fmt: off
# Run this as the FIRST cell in your Kaggle notebook.
# Paste each block into its own cell, or run the whole file as a script.

INSTALL_COMMANDS = """
# ── 1a. Install the RGBD-modified Ultralytics fork ──────────────────────────
pip uninstall ultralytics -y 2>/dev/null
pip install git+https://github.com/panchasan/ultralytics_RGBD.git

# ── 1b. Install Depth Anything V2 dependencies ─────────────────────────────
pip install roboflow tifffile
pip install transformers>=4.40.0 pillow

# ── 1c. Clone Depth Anything V2 for raw PyTorch inference (faster on GPU) ──
git clone https://github.com/DepthAnything/Depth-Anything-V2.git /kaggle/working/Depth-Anything-V2
pip install -r /kaggle/working/Depth-Anything-V2/requirements.txt

# ── 1d. Download ViT-S checkpoint ──────────────────────────────────────────
mkdir -p /kaggle/working/Depth-Anything-V2/checkpoints
wget -q -O /kaggle/working/Depth-Anything-V2/checkpoints/depth_anything_v2_vits.pth \
    https://huggingface.co/depth-anything/Depth-Anything-V2-Small/resolve/main/depth_anything_v2_vits.pth
"""

# To execute in a notebook cell, uncomment:
import subprocess, shlex
for line in INSTALL_COMMANDS.strip().splitlines():
    line = line.strip()
    if line and not line.startswith("#"):
        subprocess.run(line, shell=True, check=True)


In [ ]:
# =============================================================================
# CELL 2 — Download Dataset from Roboflow
# =============================================================================

import os
import sys
import shutil
from pathlib import Path
from glob import glob

# ── Roboflow download ───────────────────────────────────────────────────────
from roboflow import Roboflow

DATASET_DIR = Path("/kaggle/working/dataset_roboflow")

rf = Roboflow(api_key="")
project = rf.workspace("auv26").project("anand_tub_blue_only_final")
version = project.version(2)
dataset = version.download("yolov8", location=str(DATASET_DIR))

print(f"✅ Dataset downloaded to: {DATASET_DIR}")
print(f"   Contents: {os.listdir(DATASET_DIR)}")


In [ ]:
# =============================================================================
# CELL 3 — Generate Depth Maps with Depth Anything V2 (ViT-S) on Single GPU
# =============================================================================

import os
import sys
import shutil
from pathlib import Path
import cv2
import numpy as np
import torch
import tifffile
from tqdm import tqdm

# ── 1. Add Depth Anything V2 to path ────────────────────────────────────────
sys.path.insert(0, "/kaggle/working/Depth-Anything-V2")
from depth_anything_v2.dpt import DepthAnythingV2

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}
DATASET_DIR = Path("/kaggle/working/dataset_roboflow")
RGBD_DATASET = Path("/kaggle/working/dataset_rgbd")

# ── 2. Initialize Model on Device 0 ──────────────────────────────────────────
device = "cuda:0" if torch.cuda.is_available() else "cpu"
print(f"🚀 Initializing model on device: {device}")

model_configs = {
    "vits": {"encoder": "vits", "features": 64, "out_channels": [48, 96, 192, 384]}
}
depth_model = DepthAnythingV2(**model_configs["vits"])
depth_model.load_state_dict(
    torch.load(
        "/kaggle/working/Depth-Anything-V2/checkpoints/depth_anything_v2_vits.pth",
        map_location="cpu",
    )
)
depth_model = depth_model.to(device).eval()
print("✅ Model loaded successfully.")

# ── 3. Find Dataset Splits ───────────────────────────────────────────────────
roboflow_root = DATASET_DIR
candidate_dirs = [roboflow_root] + [d for d in roboflow_root.iterdir() if d.is_dir()]
dataset_root = next((d for d in candidate_dirs if (d / "train").is_dir()), None)

if dataset_root is None:
    for d in roboflow_root.rglob("train"):
        if (d / "images").is_dir():
            dataset_root = d.parent
            break

assert dataset_root is not None, f"Cannot find train/ split under {roboflow_root}"
print(f"📂 Dataset root: {dataset_root}")

SPLITS = [name for name in ["train", "valid", "val", "test"] 
          if (dataset_root / name).is_dir() and (dataset_root / name / "images").is_dir()]
print(f"📂 Detected splits: {SPLITS}")

# ── 4. Process Images Sequentially ───────────────────────────────────────────
for split in SPLITS:
    out_split = "val" if split == "valid" else split
    split_dir = dataset_root / split
    img_src = split_dir / "images"
    lbl_src = split_dir / "labels"
    
    img_dst = RGBD_DATASET / out_split / "images"
    lbl_dst = RGBD_DATASET / out_split / "labels"
    img_dst.mkdir(parents=True, exist_ok=True)
    lbl_dst.mkdir(parents=True, exist_ok=True)

    image_files = sorted(p for p in img_src.iterdir() if p.suffix.lower() in IMAGE_EXTENSIONS)
    print(f"Processing {split} split ({len(image_files)} images)...")

    for img_path in tqdm(image_files, desc=f"Split: {split}"):
        # ── Read RGB ──
        bgr = cv2.imread(str(img_path), cv2.IMREAD_COLOR)
        if bgr is None:
            continue
        rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)  # (H, W, 3) uint8

        # ── Generate depth ──
        with torch.no_grad():
            depth = depth_model.infer_image(bgr)  # returns (H, W) float

        # Normalize depth to [0, 255] uint8
        d_min, d_max = depth.min(), depth.max()
        if d_max - d_min > 1e-6:
            depth_norm = ((depth - d_min) / (d_max - d_min) * 255).astype(np.uint8)
        else:
            depth_norm = np.zeros_like(depth, dtype=np.uint8)

        # ── Stack RGBD (H, W, 4) ──
        rgbd = np.dstack((rgb, depth_norm))  # (H, W, 4) uint8

        # ── Save as 4-channel TIFF ──
        out_name = img_path.stem + ".tiff"
        tifffile.imwrite(str(img_dst / out_name), rgbd)

        # ── Copy label ──
        lbl_file = lbl_src / (img_path.stem + ".txt")
        if lbl_file.exists():
            shutil.copy2(lbl_file, lbl_dst / lbl_file.name)

# ── 5. Clean Up GPU Memory ───────────────────────────────────────────────────
# del depth_model
# torch.cuda.empty_cache()
# print("\n✅ All depth maps generated and merged successfully! GPU memory cleared.")

import gc

# after all depth maps are generated in Cell 3
del depth_model
gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()

print(torch.cuda.memory_allocated() / 1e9, "GB still allocated on GPU 0")

In [ ]:

# =============================================================================
# CELL 4 — Create data.yaml and model YAML for 4-channel YOLOv8x
# =============================================================================

import yaml

# ── Read class names from Roboflow's data.yaml ──────────────────────────────
roboflow_yaml_candidates = list(dataset_root.glob("data.yaml")) + list(
    dataset_root.glob("*.yaml")
)
roboflow_yaml = roboflow_yaml_candidates[0] if roboflow_yaml_candidates else None

if roboflow_yaml and roboflow_yaml.exists():
    with open(roboflow_yaml) as f:
        rf_data = yaml.safe_load(f)
    class_names = rf_data.get("names", {})
    nc = len(class_names) if isinstance(class_names, (list, dict)) else rf_data.get("nc", 1)
    if isinstance(class_names, list):
        class_names = {i: name for i, name in enumerate(class_names)}
    print(f"✅ Classes from Roboflow: {class_names}")
else:
    # Fallback: detect from label files
    print("⚠ No data.yaml found, attempting auto-detection of class count...")
    all_labels = list((RGBD_DATASET / "train" / "labels").glob("*.txt"))
    max_cls = 0
    for lf in all_labels[:200]:
        for line in lf.read_text().strip().splitlines():
            parts = line.split()
            if parts:
                max_cls = max(max_cls, int(parts[0]))
    nc = max_cls + 1
    class_names = {i: f"class_{i}" for i in range(nc)}
    print(f"  Auto-detected {nc} classes")

# ── Write data.yaml ──────────────────────────────────────────────────────────
data_yaml_path = RGBD_DATASET / "data.yaml"
data_config = {
    "path": str(RGBD_DATASET),
    "train": "train/images",
    "val": "val/images",
    "nc": nc,
    "names": class_names,
    "channels": 4,  # <-- This tells Ultralytics to use 4-channel inputs!
}
with open(data_yaml_path, "w") as f:
    yaml.dump(data_config, f, default_flow_style=False, sort_keys=False)
print(f"✅ data.yaml written to {data_yaml_path}")

# ── Write 4-channel YOLOv8x model config ────────────────────────────────────
# Using the 'x' (extra-large) variant: ~68M params (3ch), grows slightly with
# 4ch input → closest to the ~100M target the user mentioned.
# Scale factors: x: [1.00, 1.25, 512]

model_yaml_content = f"""\
# YOLOv8x-RGBD — 4-channel (RGBD) detection model
# Channel values pre-computed for x-scale (width_multiple=1.25, max=512)

# Parameters
nc: {nc}  # number of classes
ch: 4     # 4 input channels (RGB + Depth)

# YOLOv8x backbone (pre-scaled: no 'scales' section needed)
backbone:
  # [from, repeats, module, args]
  - [-1, 1, Conv, [80, 3, 2]]         # 0-P1/2   (64×1.25=80)
  - [-1, 1, Conv, [160, 3, 2]]        # 1-P2/4   (128×1.25=160)
  - [-1, 3, C2f, [160, True]]
  - [-1, 1, Conv, [320, 3, 2]]        # 3-P3/8   (256×1.25=320)
  - [-1, 6, C2f, [320, True]]
  - [-1, 1, Conv, [512, 3, 2]]        # 5-P4/16  (512×1.25=640→capped 512)
  - [-1, 6, C2f, [512, True]]
  - [-1, 1, Conv, [512, 3, 2]]        # 7-P5/32  (1024×1.25=1280→capped 512)
  - [-1, 3, C2f, [512, True]]
  - [-1, 1, SPPF, [512, 5]]           # 9

# YOLOv8x head
head:
  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  - [[-1, 6], 1, Concat, [1]]         # cat backbone P4
  - [-1, 3, C2f, [512]]               # 12

  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  - [[-1, 4], 1, Concat, [1]]         # cat backbone P3
  - [-1, 3, C2f, [320]]               # 15 (P3/8-small)

  - [-1, 1, Conv, [320, 3, 2]]
  - [[-1, 12], 1, Concat, [1]]        # cat head P4
  - [-1, 3, C2f, [512]]               # 18 (P4/16-medium)

  - [-1, 1, Conv, [512, 3, 2]]
  - [[-1, 9], 1, Concat, [1]]         # cat head P5
  - [-1, 3, C2f, [512]]               # 21 (P5/32-large)

  - [[15, 18, 21], 1, Detect, [nc]]   # Detect(P3, P4, P5)
"""

model_yaml_path = RGBD_DATASET / "yolov8x_rgbd.yaml"
with open(model_yaml_path, "w") as f:
    f.write(model_yaml_content)
print(f"✅ Model config written to {model_yaml_path}")


In [ ]:

print(torch.cuda.memory_allocated() / 1e9, "GB still allocated on GPU 0")

In [ ]:
# =============================================================================
# CELL 5 — Train YOLOv8x-RGBD (Single GPU)
# =============================================================================

import os

# ── 1. FORCE SINGLE GPU VISIBILITY ───────────────────────────────────────────
# This completely hides the second GPU so Ultralytics cannot auto-launch DDP
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["WANDB_PROJECT"] = "yolov8x_rgbd_train"
os.environ["OMP_NUM_THREADS"] = "4"

import torch
from ultralytics import YOLO
from ultralytics.nn.tasks import DetectionModel
import wandb

wandb.login(key=os.environ.get("WANDB_API_KEY"))

# 2. Fix PyTorch 2.6+ unpickling error
if not hasattr(torch, "_original_torch_load_patched"):
    torch._original_torch_load_patched = torch.load
    torch.load = lambda *args, **kwargs: torch._original_torch_load_patched(*args, **{**kwargs, "weights_only": False})

# 3. Force PyTorch model creation with 4 channels
print("\n" + "=" * 60)
print("🔍 VALIDATING MODEL ARCHITECTURE")
print("=" * 60)
pt_model = DetectionModel(cfg=str(model_yaml_path), ch=4, nc=nc)
if pt_model.model[0].conv.weight.shape[1] != 4:
    raise RuntimeError("Model failed to build with 4 channels!")
print(f"✅ Model successfully initialized with 4 channels. First layer shape: {pt_model.model[0].conv.weight.shape}")

# 4. Initialize YOLO wrapper
model = YOLO(str(model_yaml_path))
model.ckpt_path = None
model.model = pt_model

print(torch.cuda.memory_allocated() / 1e9, "GB still allocated on GPU 0")
print(model.model.model[0].conv.weight.shape)



print(torch.cuda.memory_allocated() / 1e9, "GB still allocated on GPU 0")


In [ ]:
def check_channels(trainer):
    ch = trainer.model.model[0].conv.weight.shape
    print(f"🔍 Trainer's model first-conv shape: {ch}")

model.add_callback("on_train_start", check_channels)
# 5. Train on Single GPU
import ultralytics.engine.trainer as trainer_module
trainer_module.check_amp = lambda model: True
print("\n🚀 STARTING SINGLE-GPU TRAINING...")
import ray.tune
if not hasattr(ray.tune, "is_session_enabled"):
    ray.tune.is_session_enabled = lambda: False

results = model.train(
    data=str(data_yaml_path),
    epochs=100,
    imgsz=640,
    batch=4,                  # Fit within single T4 16GB VRAM
    device=0,                  # Safe now that CUDA_VISIBLE_DEVICES is 0
    workers=1,
    patience=15,
    save=True,
    save_period=10,
    pretrained=False,
    optimizer="AdamW",
    lr0=0.001,
    lrf=0.01,
    warmup_epochs=5,
    warmup_momentum=0.8,
    weight_decay=0.0005,
    cos_lr=True,
    amp=True,
    cache=True,
    close_mosaic=10,
    project="runs",
    name="yolov8x_rgbd_run_single",
    exist_ok=True,
    verbose=True,
)

print("\n" + "=" * 60)
print("✅ TRAINING COMPLETE")
print("=" * 60)
print(f"   Best weights: /kaggle/working/runs/yolov8x_rgbd_run_single/weights/best.pt")
print(f"   Last weights: /kaggle/working/runs/yolov8x_rgbd_run_single/weights/last.pt")

In [ ]:
# import subprocess
# print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)

In [ ]:
# =============================================================================
# CELL 6 — Validate & Export Results
# =============================================================================

# ── Validate on val set ──────────────────────────────────────────────────────
best_model = YOLO("/kaggle/working/runs/yolov8x_rgbd_train/weights/best.pt")
metrics = best_model.val(data=str(data_yaml_path), device=0, imgsz=640)

print("\n" + "=" * 60)
print("VALIDATION RESULTS")
print("=" * 60)
print(f"   mAP50:     {metrics.box.map50:.4f}")
print(f"   mAP50-95:  {metrics.box.map:.4f}")
print(f"   Precision: {metrics.box.mp:.4f}")
print(f"   Recall:    {metrics.box.mr:.4f}")

# ── Copy best weights to /kaggle/working/ for easy download ──────────────────
shutil.copy2(
    "/kaggle/working/runs/yolov8x_rgbd_train/weights/best.pt",
    "/kaggle/working/best_rgbd.pt",
)
print("\n✅ Best weights copied to /kaggle/working/best_rgbd.pt")
print("   Download this file from the Kaggle Output tab.")

In [ ]:
# import os
# ckpt_path = "/kaggle/working/runs/yolov8x_rgbd_run_single/weights/last.pt"
# print(os.path.exists(ckpt_path))

In [ ]:
# # =============================================================================
# # RESUME TRAINING — YOLOv8x-RGBD (4-channel) from last checkpoint
# # =============================================================================

# import os
# import ray.tune
# import ultralytics.engine.trainer as trainer_module
# from ultralytics import YOLO

# ckpt_path = "/kaggle/working/runs/yolov8x_rgbd_run_single/weights/last.pt"
# assert os.path.exists(ckpt_path), f"Checkpoint not found at {ckpt_path}"

# # ── Re-apply fixes (needed again since resuming builds a fresh trainer) ─────
# trainer_module.check_amp = lambda model: True
# if not hasattr(ray.tune, "is_session_enabled"):
#     ray.tune.is_session_enabled = lambda: False

# # ── Load checkpoint directly (4-channel arch is preserved inside the .pt) ───
# model = YOLO(ckpt_path)

# # ── Sanity check: confirm first conv layer still expects 4 channels ─────────
# first_conv_shape = model.model.model[0].conv.weight.shape
# print(f"🔍 Resumed model's first-conv shape: {first_conv_shape}")
# assert first_conv_shape[1] == 4, "❌ Checkpoint did not preserve 4-channel input!"
# print("✅ Confirmed 4-channel RGBD model.")

# # ── Resume training ───────────────────────────────────────────────────────
# # resume=True pulls epochs/batch/lr/etc. from the run's saved args.yaml —
# # do NOT pass other training kwargs here, they'll be ignored or error out.
# print("\n🚀 RESUMING TRAINING FROM LAST CHECKPOINT...")
# results = model.train(resume=True)

# print("\n" + "=" * 60)
# print("✅ TRAINING COMPLETE")
# print("=" * 60)
# print(f"   Best weights: /kaggle/working/runs/yolov8x_rgbd_run_single/weights/best.pt")
# print(f"   Last weights: /kaggle/working/runs/yolov8x_rgbd_run_single/weights/last.pt")
